# 09 - Model Training and Walk-Forward Backtest
Train logistic regression, random forest, and XGBoost models via walk-forward backtesting across 5 seasonal folds with isotonic calibration.

In [ ]:
import sys
import logging
from pathlib import Path

# Ensure src/ is importable
project_root = str(Path.cwd().parent) if Path.cwd().name == 'notebooks' else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.INFO)

import pandas as pd
from src.models.backtest import run_all_models, FOLD_MAP
from src.models.feature_sets import FULL_FEATURE_COLS, CORE_FEATURE_COLS

## Configuration

In [ ]:
FEATURE_MATRIX_PATH = Path(project_root) / 'data' / 'features' / 'feature_matrix.parquet'
RESULTS_DIR = Path(project_root) / 'data' / 'results'
RESULTS_PATH = RESULTS_DIR / 'backtest_results.parquet'

## Load Feature Matrix

In [ ]:
df = pd.read_parquet(FEATURE_MATRIX_PATH)
print(f"Feature matrix: {len(df)} games, {df['season'].nunique()} seasons ({df['season'].min()}-{df['season'].max()})")
print(f"Full feature set: {len(FULL_FEATURE_COLS)} columns")
print(f"Core feature set: {len(CORE_FEATURE_COLS)} columns")

## Walk-Forward Backtest
Fold structure: train 2015..N-2, calibrate N-1, test N. Test years: 2019, 2021, 2022, 2023, 2024 (2020 skipped as test year).

In [ ]:
print("Fold map:")
for test_year, train_years, cal_year in FOLD_MAP:
    print(f"  Test {test_year}: train {train_years[0]}-{train_years[-1]}, calibrate {cal_year}")

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if RESULTS_PATH.exists():
    print(f"Results already cached at {RESULTS_PATH}")
    results_df = pd.read_parquet(RESULTS_PATH)
    print(f"Loaded {len(results_df)} predictions from cache")
else:
    print("Running walk-forward backtest for all models and feature sets...")
    print("This may take several minutes...")
    results_df = run_all_models(df)
    results_df.to_parquet(RESULTS_PATH, index=False, compression="snappy")
    print(f"Saved {len(results_df)} predictions to {RESULTS_PATH}")

## Results Summary

In [ ]:
print(f"Total predictions: {len(results_df)}")
print(f"\nModels: {results_df['model_name'].unique().tolist()}")
print(f"Feature sets: {results_df['feature_set'].unique().tolist()}")
print(f"Test years: {sorted(results_df['fold_test_year'].unique().tolist())}")
print(f"\nPredictions per model x feature set:")
print(results_df.groupby(['model_name', 'feature_set']).size().unstack(fill_value=0))

## Quick Brier Score Preview

In [ ]:
from src.models.evaluate import compute_brier_scores

brier_df = compute_brier_scores(results_df)
agg = brier_df[brier_df['fold_test_year'] == 'aggregate'].sort_values('brier_score')
print("Aggregate Brier Scores (lower is better):")
print(agg[['model_name', 'feature_set', 'brier_score', 'n_games']].to_string(index=False))